# Colormap tuner

Take a colormap (octarine, dusk, ...) and keep its **sequence of colours**, but tune its **lightness** (start, end,
and how it bends), **hue** (a rotation of all its colours) and **colourfulness**, live with sliders. Everything is
done in CAM02-UCS (J' lightness, a' b' colour), a perceptually uniform space; the logic is in `replay/cmaptune.py`.

The figure shows the tuned map next to the original, `cmyt.arbre-0.8` (f_ret) and `cgbreak_warm_v2` (log X), their
lightness, hue and colourfulness along the map, how different the tuned map is from those two **at the same
lightness** (CAM02-UCS a' b' distance; above about 10 is clearly different), how much of it falls outside the screen
gamut (clipped), and some mock orbits drawn with it. The line printed under the figure recreates the map exactly:
copy it when you like one.

Runs anywhere (laptop or jotun): no simulation data.

In [ ]:
# Loading

%matplotlib inline
import sys
from pathlib import Path

HERE = Path.cwd().resolve()            # replay/ (the notebooks' module) sits next to this notebook
if HERE.name != 'notebooks' and (HERE / 'notebooks' / 'replay').is_dir():
    HERE = HERE / 'notebooks'
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import Image, display
import io
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap
import cmasher  # noqa: F401   (registers the cmr.* colormaps)
import cmyt     # noqa: F401   (registers the cmyt.* colormaps)

from replay import cmaptune as CT     # retune, profile, distances
from replay import simdata as SD      # colormap(): names, and 'name-T' cuts
from replay import simmap as SM       # noqa: F401  (registers cgbreak_warm_v2)
from replay import plotstyle as S

ARBRE = SD.colormap('cmyt.arbre-0.8')
CGB = SD.colormap('cgbreak_warm_v2')
CGB_BLUE = ListedColormap(CGB(np.linspace(0.0, 0.499, 128)), name='cgbreak blue leg')
CGB_WARM = ListedColormap(CGB(np.linspace(0.502, 1.0, 128)), name='cgbreak warm leg')

In [ ]:
# Plotting  ·  tuner (sliders on the left, figure on the right)

# ---- controls ---------------------------------------------------------------
base    = W.Dropdown(options=['cmyt.octarine', 'cmyt.dusk', 'cmyt.arbre', 'cmyt.kelp', 'cmr.ghostlight', 'cmr.amber', 'plasma', 'viridis'],
                     value='cmyt.octarine', description='base map')
window  = W.FloatRangeSlider(value=(0.15, 0.99), min=0.0, max=1.0, step=0.01, description='window',
                             tooltip='the part of the base map that is used')
own_J   = W.Checkbox(value=False, description="keep the base's lightness")
J_start = W.FloatSlider(value=CT.ARBRE_J[0], min=0, max=100, step=0.5, description="J' start")
J_end   = W.FloatSlider(value=CT.ARBRE_J[1], min=0, max=100, step=0.5, description="J' end")
gamma   = W.FloatSlider(value=1.0, min=0.4, max=2.5, step=0.05, description='bend',
                        tooltip='< 1 lifts the dark half, > 1 deepens it')
hue     = W.FloatSlider(value=0.0, min=-180, max=180, step=1, description='hue shift')
chroma  = W.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.02, description='colourful')
reverse = W.Checkbox(value=False, description='reverse')
orbits  = W.Checkbox(value=True, description='mock orbits')
match   = W.Button(description="arbre's lightness", tooltip="J' 27.2 -> 81.3, as cmyt.arbre-0.8")

def use_arbre_lightness(_):
    own_J.value = False
    J_start.value, J_end.value = CT.ARBRE_J

match.on_click(use_arbre_lightness)
for w in (base, window, J_start, J_end, gamma, hue, chroma):
    w.style.description_width = '90px'
    w.layout.width = '330px'


# ---- the figure, redrawn on every change ------------------------------------
def draw(base, window, own_J, J_start, J_end, gamma, hue, chroma, reverse, orbits):
    cm = CT.retune(base, window=window, J=None if own_J else (J_start, J_end), gamma=gamma, hue=hue,
                   chroma=chroma, reverse=reverse, name='tuned')
    orig = ListedColormap(SD.colormap(base)(np.linspace(*window, 256)), name=f'{base} (window)')
    v = np.linspace(0, 1, 256)

    S.use()
    fig = plt.figure(figsize=(9.5, 7.2))
    bands = [('tuned', cm), (f'{base}, {window[0]:.2f}-{window[1]:.2f}', orig),
             ('cmyt.arbre-0.8 (f_ret)', ARBRE), ('cgbreak_warm_v2 (log X)', CGB)]
    gs = fig.add_gridspec(len(bands) + 2, 3, height_ratios=[0.5] + [0.3] * (len(bands) - 1) + [0.12, 2.3],
                          hspace=0.45, wspace=0.3)
    for i, (label, m) in enumerate(bands):
        ax = fig.add_subplot(gs[i, :])
        ax.imshow(m(v)[None, :, :3], aspect='auto')
        ax.set_axis_off()
        ax.text(-0.01, 0.5, label, transform=ax.transAxes, ha='right', va='center', fontsize=8)

    # lightness, hue, colourfulness along the map
    axJ, axH, axO = (fig.add_subplot(gs[-1, k]) for k in range(3))
    for m, colour, ls, lw in ((cm, '#111111', '-', 1.8), (orig, '#888888', '--', 1.0), (ARBRE, '#4c94dc', ':', 1.2)):
        p = CT.profile(m)
        axJ.plot(p['v'], p['J'], color=colour, ls=ls, lw=lw, label=m.name if m is not cm else 'tuned')
        axH.plot(p['v'], p['hue'], '.', ms=1.4, color=colour)
    axJ.set_xlabel('value along the map'); axJ.set_ylabel("lightness J'"); axJ.set_ylim(0, 100)
    axJ.legend(fontsize=6, frameon=False, loc='upper left')
    axH.set_xlabel('value along the map'); axH.set_ylabel('hue [deg]'); axH.set_ylim(0, 360)

    # mock orbits drawn with the tuned map (values falling with time, as M* or f_ret do)
    if orbits:
        rng = np.random.default_rng(3)
        for i in range(7):
            t0 = 1 + 1.7 * i
            tt = np.arange(t0, 13.8, 0.01)
            rr = 1.6 * np.exp(-(tt - t0) / 5) * (0.3 + 0.7 * np.abs(np.cos(np.pi * (tt - t0) / (1.4 + 0.2 * i)))) + 0.02
            val = np.clip(0.95 - 0.09 * (tt - t0) + 0.1 * rng.normal(), 0, 1)
            pts = np.stack([tt, rr], 1)
            seg = np.stack([pts[:-1], pts[1:]], 1)
            axO.add_collection(LineCollection(seg, colors=cm(0.5 * (val[:-1] + val[1:])), linewidths=1.4))
        axO.set_xlim(0, 13.8); axO.set_yscale('log'); axO.set_ylim(1e-2, 3)
        axO.axhline(1, color='#52514e', lw=0.8, ls=(0, (3, 2)))
        axO.set_xlabel('mock time'); axO.tick_params(labelsize=6)
    else:
        axO.set_axis_off()

    # shown as a PNG (plt.show() inside a widget panel is not always forwarded by the kernel)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    display(Image(data=buf.getvalue()))

    # numbers
    p = CT.profile(cm)
    st = p['steps']
    d = {name: CT.distance_at_equal_lightness(cm, ref) for name, ref in
         (('arbre-0.8', ARBRE), ('cgbreak blue leg', CGB_BLUE), ('cgbreak warm leg', CGB_WARM))}
    print(f"J' {p['J'][0]:.1f} -> {p['J'][-1]:.1f} | monotonic {bool(np.all(np.diff(p['J']) >= -1e-6))} | "
          f"perceptual steps {st.mean():.2f} +- {st.std():.2f} | clipped to the screen gamut: {100 * cm.out_of_gamut:.0f}%")
    print('distance at equal lightness (median / minimum; > ~10 clearly different): '
          + ' | '.join(f'{k} {a:.0f} / {b:.0f}' for k, (a, b) in d.items()))
    print('recreate:  CT.' + cm.spec)


controls = dict(base=base, window=window, own_J=own_J, J_start=J_start, J_end=J_end, gamma=gamma, hue=hue,
                chroma=chroma, reverse=reverse, orbits=orbits)
out = W.Output()

def update(_=None):
    # redraw into the panel; an error is printed there instead of vanishing
    with out:
        out.clear_output(wait=True)
        try:
            draw(**{key: w.value for key, w in controls.items()})
        except Exception:
            import traceback
            traceback.print_exc()

for w in controls.values():
    w.observe(update, names='value')

panel = W.VBox([base, window, own_J, J_start, J_end, match, gamma, hue, chroma, reverse, orbits],
               layout=W.Layout(width='350px'))
display(W.HBox([panel, out]))
update()

**Tips.** *window* picks the part of the base map you use (octarine 0.15–0.99 has arbre's lightness range).
*J' start / end* set the lightness at the two ends (the button puts arbre-0.8's 27.2 → 81.3); *bend* curves the
lightness ramp. *hue shift* turns every colour around the wheel by the same angle (the sequence stays), and
*colourful* scales saturation: above 1 it soon runs out of screen gamut (watch the clipped %). When you have one you
like, copy the *recreate* line: I can make it a named map (e.g. for `MSTAR['cmap']`).

## Diverging maps: the width of the middle band

For maps with a dark (or light) centre, e.g. `cmr.iceburn`: **middle** < 1 leaves the centre faster, so the middle band gets narrower (iceburn's black), > 1 widens it; both ends stay. **cut** shows the 'name-T' version (0 → T of the map, as `'iceburn-custom-0.9'`; on a diverging map this also moves the centre). The *recreate* line under the figure gives the map; `iceburn-custom` (registered, `replay/custom_cmaps.py`) is `squeeze('cmr.iceburn', middle=0.5)`: send me your line to replace it.

In [ ]:
# Plotting  ·  diverging tuner: the middle band (sliders on the left, figure on the right)

# ---- controls ---------------------------------------------------------------
d_base   = W.Dropdown(options=['cmr.iceburn', 'iceburn-custom', 'cmr.redshift', 'cmr.fusion', 'cmr.wildfire',
                               'RdBu', 'coolwarm'], value='cmr.iceburn', description='base map')
d_middle = W.FloatSlider(value=0.5, min=0.15, max=2.0, step=0.05, description='middle',
                         tooltip='< 1 narrows the middle band, > 1 widens it')
d_cut    = W.FloatSlider(value=1.0, min=0.5, max=1.0, step=0.01, description='cut (0.X)',
                         tooltip="the 'name-T' version: 0 -> T of the map")
d_rev    = W.Checkbox(value=False, description='reverse')
d_orbits = W.Checkbox(value=True, description='mock orbits')
for w in (d_base, d_middle, d_cut):
    w.style.description_width = '90px'
    w.layout.width = '330px'


# ---- the figure, redrawn on every change ------------------------------------
def draw_div(base, middle, cut, reverse, orbits):
    sq = CT.squeeze(base, middle=middle, name='squeezed')
    if reverse:
        sq = sq.reversed()
    orig = SD.colormap(base)
    cut_map = ListedColormap(sq(np.linspace(0.0, cut, 256)), name=f'squeezed-{cut:g}')
    v = np.linspace(0, 1, 256)

    S.use()
    fig = plt.figure(figsize=(9.5, 5.6))
    bands = [('squeezed', sq), (f'{base}', orig), (f'squeezed, cut {cut:g}', cut_map)]
    gs = fig.add_gridspec(len(bands) + 2, 2, height_ratios=[0.5, 0.3, 0.3, 0.12, 2.3], hspace=0.45, wspace=0.25)
    for i, (label, m) in enumerate(bands):
        ax = fig.add_subplot(gs[i, :])
        ax.imshow(m(v)[None, :, :3], aspect='auto')
        ax.set_axis_off()
        ax.text(-0.01, 0.5, label, transform=ax.transAxes, ha='right', va='center', fontsize=8)

    # lightness along the map: the dip (or peak) at the centre gets narrower with middle < 1
    axJ, axO = fig.add_subplot(gs[-1, 0]), fig.add_subplot(gs[-1, 1])
    for m, colour, ls, lw, lab in ((sq, '#111111', '-', 1.8, 'squeezed'), (orig, '#888888', '--', 1.0, base)):
        p = CT.profile(m)
        axJ.plot(p['v'], p['J'], color=colour, ls=ls, lw=lw, label=lab)
    axJ.set_xlabel('value along the map'); axJ.set_ylabel("lightness J'"); axJ.set_ylim(0, 100)
    axJ.legend(fontsize=6, frameon=False, loc='upper center')

    # mock orbits coloured by a ratio around 1 (as Comp02b's), on 0.98 .. 1.02
    if orbits:
        rng = np.random.default_rng(5)
        for i in range(7):
            t0 = 1 + 1.7 * i
            tt = np.arange(t0, 13.8, 0.01)
            rr = 1.6 * np.exp(-(tt - t0) / 5) * (0.3 + 0.7 * np.abs(np.cos(np.pi * (tt - t0) / (1.4 + 0.2 * i)))) + 0.02
            val = 1 + 0.015 * np.tanh((tt - t0 - 2) / 2) * rng.choice([-1, 0, 1]) + 0.004 * rng.normal()
            x = np.clip((val - 0.98) / 0.04, 0, 1)
            pts = np.stack([tt, rr], 1)
            seg = np.stack([pts[:-1], pts[1:]], 1)
            axO.add_collection(LineCollection(seg, colors=cut_map(0.5 * (x[:-1] + x[1:])), linewidths=1.4))
        axO.set_xlim(0, 13.8); axO.set_yscale('log'); axO.set_ylim(1e-2, 3)
        axO.axhline(1, color='#52514e', lw=0.8, ls=(0, (3, 2)))
        axO.set_xlabel('mock time (ratio 0.98 .. 1.02)'); axO.tick_params(labelsize=6)
    else:
        axO.set_axis_off()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    display(Image(data=buf.getvalue()))
    print('recreate:  CT.' + sq.spec + ('  (reversed: append _r to its name)' if reverse else '')
          + (f"   |   cut: '<name>-{cut:g}'" if cut < 1 else ''))


d_controls = dict(base=d_base, middle=d_middle, cut=d_cut, reverse=d_rev, orbits=d_orbits)
d_out = W.Output()

def d_update(_=None):
    with d_out:
        d_out.clear_output(wait=True)
        try:
            draw_div(**{key: w.value for key, w in d_controls.items()})
        except Exception:
            import traceback
            traceback.print_exc()

for w in d_controls.values():
    w.observe(d_update, names='value')

display(W.HBox([W.VBox([d_base, d_middle, d_cut, d_rev, d_orbits], layout=W.Layout(width='350px')), d_out]))
d_update()